# Fresh Risk Detection Inference - Images and Videos

Self-contained inference notebook. It loads from:

`/content/drive/MyDrive/risk_detection_fresh_outputs/`

It outputs annotated images/videos and a video event CSV with timestamps. Includes context-aware downgrading for kitchen knives, bonfire/social alcohol/tobacco/vape scenes, and non-threatening utility use.

In [ ]:
# CELL 1 | Install dependencies, mount Drive
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "ultralytics>=8.3.0" opencv-python-headless pandas pyyaml

import subprocess, torch
try:
    r=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
    print('GPU:', r.stdout.strip() if r.returncode==0 else 'No GPU visible; CPU inference works but video is slower.')
except Exception as exc: print('GPU check skipped:', exc)


In [ ]:
# CELL 2 | Load fresh risk detector
from pathlib import Path
import json, yaml, shutil, time
import cv2, numpy as np, pandas as pd
from ultralytics import YOLO
from IPython.display import display, Image as IPyImage, HTML
from base64 import b64encode

DRIVE_OUT = Path('/content/drive/MyDrive/risk_detection_fresh_outputs')
if not DRIVE_OUT.exists():
    candidates=list(Path('/content/drive/MyDrive').glob('**/risk_detection_fresh_outputs'))
    if candidates: DRIVE_OUT=candidates[0]
WEIGHTS_PATH=DRIVE_OUT/'best_risk_detector.pt'
DATA_YAML=DRIVE_OUT/'risk_data.yaml'
OUT_DIR=Path('/content/risk_detection_fresh_test_outputs'); OUT_DIR.mkdir(parents=True, exist_ok=True)
if not WEIGHTS_PATH.exists(): raise FileNotFoundError(f'Missing weights: {WEIGHTS_PATH}')
if DATA_YAML.exists():
    data=yaml.safe_load(DATA_YAML.read_text()) or {}; names=data.get('names',['firearm','ammo','knife_blade','alcohol','tobacco_vape']); RISK_CLASSES=list(names.values()) if isinstance(names,dict) else list(names)
else: RISK_CLASSES=['firearm','ammo','knife_blade','alcohol','tobacco_vape']
model=YOLO(str(WEIGHTS_PATH)); object_model=YOLO('yolov8n.pt')
print('Loaded:', WEIGHTS_PATH); print('Risk classes:', RISK_CLASSES)


In [ ]:
# CELL 3 | Risk detector with timestamps, ammo/tobacco/vape, and benign context handling
RISK_COLORS={'firearm':(0,0,255),'ammo':(40,40,255),'knife_blade':(0,80,255),'alcohol':(0,180,255),'tobacco_vape':(255,0,255),'benign_context_knife':(160,160,160),'benign_context_alcohol':(160,160,160),'benign_context_tobacco_vape':(160,160,160)}
KITCHEN_CONTEXT={'bowl','spoon','fork','cup','dining table','carrot','broccoli','banana','apple','sandwich','pizza','cake','oven','microwave','refrigerator','sink'}
SOCIAL_BENIGN_CONTEXT={'person','cup','bottle','wine glass','dining table','chair','bench'}
FIRE_BONFIRE_CONTEXT={'fire hydrant'}  # COCO has no bonfire/fire class; this is intentionally conservative.
RISK_LEVELS={'none':0,'low':1,'medium':2,'high':3,'critical':4}
CLASS_CONF={'firearm':0.35,'ammo':0.35,'knife_blade':0.35,'alcohol':0.35,'tobacco_vape':0.30}

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b; ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); aa=max(1,(ax2-ax1)*(ay2-ay1)); bb=max(1,(bx2-bx1)*(by2-by1)); return inter/(aa+bb-inter+1e-6)

def dist(a,b,w,h):
    ax=(a[0]+a[2])/2; ay=(a[1]+a[3])/2; bx=(b[0]+b[2])/2; by=(b[1]+b[3])/2; return (((ax-bx)**2+(ay-by)**2)**0.5)/max(1,(w*w+h*h)**0.5)

def context(frame):
    out=[]
    try:
        res=object_model(frame,imgsz=640,conf=0.25,verbose=False)[0]
        for b in res.boxes:
            cid=int(b.cls[0]); out.append({'label':res.names.get(cid,str(cid)),'confidence':float(b.conf[0]),'bbox':[int(v) for v in b.xyxy[0].cpu().numpy().tolist()]})
    except Exception as exc: print('Context skipped:', exc)
    return out

def near_any(box, ctx, labels, w, h, max_dist=0.30, min_iou=0.0):
    return any(o['label'] in labels and (dist(box,o['bbox'],w,h)<max_dist or iou(box,o['bbox'])>min_iou) for o in ctx)

def risk_for(label, conf, box, shape, ctx):
    h,w=shape[:2]
    person_near=near_any(box,ctx,{'person'},w,h,max_dist=0.25,min_iou=0.01)
    kitchen_near=near_any(box,ctx,KITCHEN_CONTEXT,w,h,max_dist=0.28)
    social_near=near_any(box,ctx,SOCIAL_BENIGN_CONTEXT,w,h,max_dist=0.35)
    if label=='knife_blade':
        if kitchen_near and not person_near and conf<0.75:
            return 'benign_context_knife','low','Knife appears in kitchen/cooking context; downgraded.'
        return label, 'high' if conf>=0.60 else 'medium','Knife/blade detected.'
    if label=='firearm': return label,'critical' if conf>=0.55 else 'high','Firearm-like object detected.'
    if label=='ammo': return label,'high' if conf>=0.55 else 'medium','Bullets/ammunition detected.'
    if label=='alcohol':
        if social_near and conf<0.65:
            return 'benign_context_alcohol','low','Alcohol appears in social/dining context; downgraded but logged.'
        return label,'medium' if conf>=0.55 else 'low','Alcohol container detected.'
    if label=='tobacco_vape':
        if social_near and conf<0.60:
            return 'benign_context_tobacco_vape','low','Tobacco/vape appears in social context; downgraded but logged.'
        return label,'medium' if conf>=0.50 else 'low','Tobacco/vape/smoking detected.'
    return label,'low','Risk object detected.'

def overall(levels): return max(levels, key=lambda x:RISK_LEVELS.get(x,0)) if levels else 'none'

class RiskDetector:
    def __init__(self, model, conf=0.25, imgsz=640): self.model=model; self.conf=conf; self.imgsz=imgsz
    def predict_frame(self, frame):
        t0=time.perf_counter(); ctx=context(frame); res=self.model(frame,imgsz=self.imgsz,conf=self.conf,verbose=False)[0]; dets=[]
        for b in res.boxes:
            cid=int(b.cls[0]); label=res.names.get(cid,RISK_CLASSES[cid] if cid<len(RISK_CLASSES) else str(cid)); conf=float(b.conf[0])
            if conf<CLASS_CONF.get(label,self.conf): continue
            box=[int(v) for v in b.xyxy[0].cpu().numpy().tolist()]; final,level,reason=risk_for(label,conf,box,frame.shape,ctx)
            dets.append({'label':final,'raw_label':label,'confidence':conf,'bbox':box,'risk_level':level,'reason':reason})
        ov=overall([d['risk_level'] for d in dets if not d['label'].startswith('benign_context_')])
        return {'detections':dets,'overall_risk':ov,'_meta':{'latency_ms':round((time.perf_counter()-t0)*1000,1)}}, self.annotate(frame,dets,ov)
    def annotate(self, frame, dets, ov):
        out=frame.copy(); cv2.putText(out,f'Overall risk: {ov}',(16,32),cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255) if ov in ['high','critical'] else (0,180,255),2)
        for d in dets:
            x1,y1,x2,y2=d['bbox']; color=RISK_COLORS.get(d['label'],(0,255,0)); cv2.rectangle(out,(x1,y1),(x2,y2),color,2); cv2.putText(out,f"{d['label']} {d['confidence']:.2f} {d['risk_level']}",(x1,max(18,y1-6)),cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
        return out
    def run_image(self, path):
        img=cv2.imread(str(path)); payload,ann=self.predict_frame(img); out=OUT_DIR/f'{Path(path).stem}_risk.jpg'; cv2.imwrite(str(out),ann); display(IPyImage(filename=str(out))); print(json.dumps(payload,indent=2)); return payload,out
    def run_video(self, path, target_fps=None, slow_factor=1.0, display_width=720):
        cap=cv2.VideoCapture(str(path)); fps=cap.get(cv2.CAP_PROP_FPS) or 30; total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0); w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); every=1 if target_fps is None else max(1,round(fps/target_fps)); out=OUT_DIR/f'{Path(path).stem}_risk.mp4'; csv=OUT_DIR/f'{Path(path).stem}_risk_events.csv'; writer=cv2.VideoWriter(str(out),cv2.VideoWriter_fourcc(*'mp4v'),max(1,fps/max(1.0,slow_factor)),(w,h)); rows=[]; last=None; idx=0
        while cap.isOpened():
            ok,frame=cap.read();
            if not ok: break
            if idx%every==0:
                payload,ann=self.predict_frame(frame); last=ann
                for d in payload['detections']: rows.append({'frame':idx,'time_sec':idx/fps,**d})
            else: ann=last if last is not None else frame
            writer.write(ann); idx+=1
            if idx%50==0: print(f'Processed {idx}/{total or "?"} frames')
        cap.release(); writer.release(); df=pd.DataFrame(rows); df.to_csv(csv,index=False)
        if len(df):
            summary=df.groupby(['label','risk_level']).agg(frames=('frame','nunique'),first_time_sec=('time_sec','min'),last_time_sec=('time_sec','max'),max_conf=('confidence','max')).reset_index(); display(summary)
        else: print('No risk events detected.')
        display(HTML(f'<video controls width="{display_width}"><source src="data:video/mp4;base64,{b64encode(open(out,"rb").read()).decode()}" type="video/mp4"></video>'))
        return out,csv
risk_detector=RiskDetector(model, conf=0.25, imgsz=640)
print('Fresh risk inference ready with ammo, tobacco/vape, and benign-context handling.')


In [ ]:
# CELL 4 | Upload/test images
from google.colab import files
uploaded=files.upload()
for name in uploaded.keys():
    payload,out=risk_detector.run_image(name); shutil.copy2(out, DRIVE_OUT/Path(out).name); print('Saved:', DRIVE_OUT/Path(out).name)


In [ ]:
# CELL 5 | Upload/test videos with timestamps
from google.colab import files
uploaded=files.upload()
for name in uploaded.keys():
    out,csv=risk_detector.run_video(name, target_fps=None, slow_factor=1.0, display_width=720)
    shutil.copy2(out, DRIVE_OUT/Path(out).name); shutil.copy2(csv, DRIVE_OUT/Path(csv).name)
    print('Saved video:', DRIVE_OUT/Path(out).name); print('Saved event CSV:', DRIVE_OUT/Path(csv).name)
